# Phase 2 · Model 4 · Knowledge distillation for legal NER

The Model 1 adapters + CRF checkpoint is the teacher. The student starts from Model 4's Phase 1 embeddings, first eight gated convolution blocks, and final normalization. Its bidirectional character head adds right context within each window.

Training gives equal weight to gold-label cross-entropy and KL divergence to cached teacher marginals at temperature 2. The student uses fixed teacher transition scores for decoding. The cache contains teacher probabilities for the training documents.

Cell outputs were recovered from the saved run files. Kaggle-only messages and execution numbers are unavailable in that archive.

## 1. Required files and settings

- Attach `teacher_crf/` with the CRF run's `best.pt`, `tokenizer.json`, `config.json`, `split.json`, `labels.txt`, and `evaluate_phase2.py`.
- Attach `model4_phase1/` with Model 4's Phase 1 `best.pt`, `tokenizer.json`, and `run_config.json`. Both folders are in the `model4_phase2_inputs.zip` backup.
- Enable a GPU. Attach the teacher's `Phase_2_LNER` files or enable Internet and set `HF_TOKEN` or `HF`. Dataset hashes and the split are checked against the teacher.
- For the no-KD-loss control, choose a separate `ROOT_OUT` and set `CONFIG['kd_weight'] = 0.0` in the settings cell before training.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','numpy','huggingface-hub','matplotlib'],check=True)


In [ ]:
import copy, gc, hashlib, heapq, json, math, os, random, re, shutil, time, zipfile
from collections import defaultdict
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from IPython.display import display, HTML

SOURCES = ('supreme_court', 'high_courts', 'acts', 'tribunals')
PAD, UNK, BOS, EOS = range(4)
VERSION = 'local-global-causal-v1'

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def write_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')
    os.replace(tmp, path)

def digest_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024**2), b''): h.update(block)
    return h.hexdigest()

def atomic_save(value, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(value, str(path) + '.tmp')
    os.replace(str(path) + '.tmp', path)

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(),
                torch=torch.get_rng_state(), cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])

def restore_rng(s):
    random.setstate(s['python']); np.random.set_state(s['numpy']); torch.set_rng_state(s['torch'])
    if s['cuda']: torch.cuda.set_rng_state_all(s['cuda'])

from collections import Counter
import bisect


In [ ]:
INPUT_ROOT=Path('/kaggle/input')
TEACHER_DIR=None  # Folder containing the CRF best.pt and config.json.
MODEL4_DIR=None   # Folder containing model 4 Phase-1 best.pt and tokenizer.json.
DATA_ROOT=None
RESUME_FROM=None  # Saved phase2_model4_distillation folder, if resuming.
ROOT_OUT=Path('/kaggle/working/phase2_model4_distillation')
RUN_DIR=ROOT_OUT/'student'
ASSET_DIR=ROOT_OUT/'assets'
REPO='Exploration-Lab/CS-781-Capstone'
DATA_REVISION=None
CONFIG=dict(seed=42,validation_fraction=.15,window=1024,overlap=256,epochs=40,
    effective_batch=4,micro_batch=1,activation_checkpointing=True,
    backbone_lr=5e-5,head_lr=5e-4,min_lr_ratio=.1,warmup_fraction=.06,
    weight_decay=.01,clip=1.,eval_every=1000,save_every=100,log_every=20,
    class_weight_cap=1.,head_dim=128,char_buckets=4096,bi_dropout=.1,max_hours=10.,
    student_layers=8,student_heads=8,kd_temperature=2.,kd_weight=.5,
    head_version='model4-crf-posterior-kd-v1',method='distillation')
if RESUME_FROM is not None and Path(RESUME_FROM).resolve()!=ROOT_OUT.resolve():
    if ROOT_OUT.exists() and any(ROOT_OUT.iterdir()): raise ValueError('Restore into an empty output folder')
    shutil.copytree(RESUME_FROM,ROOT_OUT)
ASSET_DIR.mkdir(parents=True,exist_ok=True)
if TEACHER_DIR is None:
    found=[]
    for p in INPUT_ROOT.rglob('config.json'):
        try:
            c=read_json(p)
            if c.get('config',{}).get('head_version')=='model1-adapters-crf-v1' and (p.parent/'best.pt').is_file(): found.append(p.parent)
        except (ValueError,OSError): pass
    if len(found)!=1: raise ValueError(f'Set TEACHER_DIR explicitly: {found}')
    TEACHER_DIR=found[0]
if MODEL4_DIR is None:
    found=[]
    for p in INPUT_ROOT.rglob('run_config.json'):
        try:
            c=read_json(p)
            if 'kernel_size' in c.get('model',{}) and (p.parent/'best.pt').is_file(): found.append(p.parent)
        except (ValueError,OSError): pass
    if len(found)!=1: raise ValueError('Set MODEL4_DIR explicitly to the uploaded model4_phase1 folder')
    MODEL4_DIR=found[0]
TEACHER_DIR=Path(TEACHER_DIR); MODEL4_DIR=Path(MODEL4_DIR)
BASE_RUN=MODEL4_DIR; BASE_CHECKPOINT=MODEL4_DIR/'best.pt'
EVALUATOR_PATH=TEACHER_DIR/'evaluate_phase2.py'
assert EVALUATOR_PATH.is_file(), 'Upload evaluate_phase2.py with the teacher'
if (RUN_DIR/'config.json').exists():
    saved=read_json(RUN_DIR/'config.json')['config']
    keys=set(CONFIG)-{'micro_batch','activation_checkpointing'}
    if any(CONFIG[k]!=saved.get(k) for k in keys): raise ValueError('Settings changed; choose a new output root')
    CONFIG=saved
print('Teacher:',TEACHER_DIR)
print('Model 4:',MODEL4_DIR)
print('Output:',ROOT_OUT)


In [ ]:
def get_data_root(explicit=None):
    needed=['Train/fold1-text.json','Train/fold1-gold.json','Train/fold2-text.json',
            'Train/fold2-gold.json','Train/labels.txt','Test/fold3-text.json']
    def valid(root): return all((root/p).is_file() for p in needed)
    if explicit is not None:
        root=Path(explicit)
        if not valid(root): raise FileNotFoundError(root)
        return root
    found=sorted({p.parent.parent for p in INPUT_ROOT.rglob('fold1-text.json') if valid(p.parent.parent)})
    if len(found)==1: return found[0]
    if len(found)>1: raise ValueError(f'Set DATA_ROOT: {found}')
    root=ASSET_DIR/'Phase_2_LNER'
    if valid(root): return root
    token=os.environ.get('HF_TOKEN')
    try:
        from kaggle_secrets import UserSecretsClient
        secrets=UserSecretsClient()
        for name in ('HF_TOKEN','HF'):
            try:
                token=secrets.get_secret(name)
                if token: break
            except Exception: pass
    except ImportError: pass
    from huggingface_hub import HfApi,hf_hub_download
    revision=HfApi().dataset_info(REPO,revision=DATA_REVISION,token=token).sha
    for name in needed:
        path=hf_hub_download(repo_id=REPO,repo_type='dataset',revision=revision,
            filename='Phase_2_LNER/'+name,token=token)
        target=root/name; target.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(path,target)
    write_json(ASSET_DIR/'dataset_revision.json',dict(repo=REPO,revision=revision))
    return root


In [ ]:
DATA_ROOT=get_data_root(DATA_ROOT)
print(DATA_ROOT)


## 2. Tokenizer and model definitions


In [ ]:
def split_bpe_pieces(text, max_chars=256):
    for match in re.finditer(' ?\\w+| ?[^\\w\\s]+|\\s+', text):
        piece = match.group(0)
        for start in range(0, len(piece), max_chars):
            yield piece[start:start + max_chars].encode('utf-8')


In [ ]:
def apply_bpe_merges(piece, merge_ranks):
    ids = [byte + 4 for byte in piece]
    size = len(ids)
    if size < 2 or not merge_ranks:
        return tuple(ids)
    previous = [i - 1 for i in range(size)]
    following = [i + 1 for i in range(size)]
    following[-1] = -1
    active = [True] * size
    queue = []

    def offer(left):
        if left < 0 or not active[left]:
            return
        right = following[left]
        if right < 0:
            return
        pair = (ids[left], ids[right])
        rank = merge_ranks.get(pair)
        if rank is not None:
            heapq.heappush(queue, (rank, left, right, pair[0], pair[1]))
    for left in range(size - 1):
        offer(left)
    while queue:
        rank, left, right, a, b = heapq.heappop(queue)
        if not active[left] or not active[right] or following[left] != right:
            continue
        if ids[left] != a or ids[right] != b:
            continue
        ids[left] = 260 + rank
        active[right] = False
        following[left] = following[right]
        if following[right] >= 0:
            previous[following[right]] = left
        offer(previous[left])
        offer(left)
    result = []
    position = 0
    while position >= 0:
        result.append(ids[position])
        position = following[position]
    return tuple(result)


In [ ]:
class ScratchByteBPE:
    FORMAT = 'handwritten-byte-bpe-v1'
    PATTERN = ' ?\\w+| ?[^\\w\\s]+|\\s+'

    def __init__(self, merges=None, max_piece_chars=256):
        self.specials = ['<pad>', '<unk>', '<s>', '</s>']
        self.max_piece_chars = max_piece_chars
        if not isinstance(max_piece_chars, int) or max_piece_chars < 1:
            raise ValueError('max_piece_chars must be a positive integer')
        self.merges = []
        self.merge_ranks = {}
        self.token_bytes = [None] * 4 + [bytes([b]) for b in range(256)]
        self._cache = {}
        for pair in merges or []:
            self.add_merge(tuple(pair))

    def add_merge(self, pair):
        if len(pair) != 2 or pair in self.merge_ranks:
            raise ValueError('Invalid or repeated merge pair')
        if any((type(token) is not int or not 4 <= token < len(self.token_bytes) for token in pair)):
            raise ValueError('A merge must reference earlier non-special token IDs')
        new_id = len(self.token_bytes)
        self.merge_ranks[pair] = len(self.merges)
        self.merges.append(pair)
        self.token_bytes.append(self.token_bytes[pair[0]] + self.token_bytes[pair[1]])
        self._cache.clear()
        return new_id

    def get_vocab_size(self):
        return len(self.token_bytes)

    def token_to_id(self, special):
        return self.specials.index(special) if special in self.specials else None

    def encode(self, text):
        if not isinstance(text, str):
            raise TypeError('encode expects a string')
        output = []
        for piece in split_bpe_pieces(text, self.max_piece_chars):
            encoded = self._cache.get(piece)
            if encoded is None:
                encoded = apply_bpe_merges(piece, self.merge_ranks)
                if len(self._cache) >= 65536:
                    self._cache.clear()
                self._cache[piece] = encoded
            output.extend(encoded)
        return SimpleNamespace(ids=output)

    def decode(self, ids, skip_special_tokens=True):
        chunks = []
        for value in ids:
            token = int(value)
            if not 0 <= token < len(self.token_bytes):
                raise ValueError(f'Unknown token ID: {token}')
            if token < 4:
                if not skip_special_tokens:
                    chunks.append(self.specials[token].encode('utf-8'))
            else:
                chunks.append(self.token_bytes[token])
        return b''.join(chunks).decode('utf-8', errors='replace')

    def save(self, path, training=None):
        payload = {'format': self.FORMAT, 'special_tokens': self.specials, 'byte_offset': 4, 'pattern': self.PATTERN, 'max_piece_chars': self.max_piece_chars, 'merges': self.merges, 'vocab_hex': [None if value is None else value.hex() for value in self.token_bytes]}
        if training is not None:
            payload['training'] = training
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(path.suffix + '.tmp')
        temporary.write_text(json.dumps(payload, ensure_ascii=False), encoding='utf-8')
        temporary.replace(path)

    @classmethod
    def from_file(cls, path):
        payload = json.loads(Path(path).read_text(encoding='utf-8'))
        if payload.get('format') != cls.FORMAT:
            raise ValueError('This is not our handwritten BPE format. Use a fresh data directory and retrain the tokenizer/token arrays.')
        if payload.get('special_tokens') != ['<pad>', '<unk>', '<s>', '</s>']:
            raise ValueError('Special-token IDs do not match')
        if payload.get('byte_offset') != 4 or payload.get('pattern') != cls.PATTERN:
            raise ValueError('Byte mapping or splitting rules do not match')
        tokenizer = cls(payload['merges'], payload['max_piece_chars'])
        expected = [None if value is None else value.hex() for value in tokenizer.token_bytes]
        if payload.get('vocab_hex') != expected:
            raise ValueError('Vocabulary is inconsistent with the merge history')
        return tokenizer


In [ ]:
class Attention(nn.Module):

    def __init__(self, width, heads, dropout):
        super().__init__()
        assert width % heads == 0
        self.heads, self.dim, self.dropout = (heads, width // heads, dropout)
        self.qkv = nn.Linear(width, 3 * width)
        self.proj = nn.Linear(width, width)

    def forward(self, x, past=None, cache=False):
        b, t, c = x.shape
        q, k, v = self.qkv(x).view(b, t, 3, self.heads, self.dim).unbind(2)
        q, k, v = (a.transpose(1, 2) for a in (q, k, v))
        if past is not None:
            if t != 1:
                raise ValueError('Cached decoding accepts one new token at a time')
            k, v = (torch.cat((past[0], k), 2), torch.cat((past[1], v), 2))
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=past is None)
        return (self.proj(y.transpose(1, 2).contiguous().view(b, t, c)), (k, v) if cache else None)

class Block(nn.Module):

    def __init__(self, width, heads, dropout):
        super().__init__()
        self.ln1, self.ln2 = (nn.LayerNorm(width), nn.LayerNorm(width))
        self.attn = Attention(width, heads, dropout)
        self.mlp = nn.Sequential(nn.Linear(width, 4 * width), nn.GELU(), nn.Linear(4 * width, width))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, past=None, cache=False):
        a, kv = self.attn(self.ln1(x), past, cache)
        x = x + self.drop(a)
        return (x + self.drop(self.mlp(self.ln2(x))), kv)


In [ ]:
class DecoderLM(nn.Module):

    def __init__(self, vocab_size=32768, width=704, layers=16, heads=11, context=2048, dropout=0.0):
        super().__init__()
        self.config = dict(vocab_size=vocab_size, width=width, layers=layers, heads=heads, context=context, dropout=dropout)
        self.token = nn.Embedding(vocab_size, width)
        self.position = nn.Embedding(context, width)
        self.blocks = nn.ModuleList([Block(width, heads, dropout) for _ in range(layers)])
        self.final_norm = nn.LayerNorm(width)
        self.drop = nn.Dropout(dropout)
        self.gradient_checkpointing = True
        self.apply(self._init)
        for block in self.blocks:
            nn.init.normal_(block.attn.proj.weight, std=0.02 / math.sqrt(2 * layers))
            nn.init.normal_(block.mlp[2].weight, std=0.02 / math.sqrt(2 * layers))

    @staticmethod
    def _init(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, ids, targets=None, past=None, cache=False):
        offset = 0 if past is None else past[0][0].shape[2]
        if offset + ids.shape[1] > self.config['context']:
            raise ValueError('Context window exceeded')
        x = self.drop(self.token(ids) + self.position(torch.arange(offset, offset + ids.shape[1], device=ids.device)))
        kvs = []
        for i, block in enumerate(self.blocks):
            if self.training and self.gradient_checkpointing:
                x = checkpoint(lambda z, b=block: b(z)[0], x, use_reentrant=False)
            else:
                x, kv = block(x, None if past is None else past[i], cache)
                if cache:
                    kvs.append(kv)
        x = self.final_norm(x)
        if targets is not None:
            flat, labels = (x.reshape(-1, x.shape[-1]), targets.reshape(-1))

            def chunk_loss(h, y):
                return F.cross_entropy(F.linear(h, self.token.weight).float(), y, ignore_index=-100, reduction='sum')
            loss = x.new_zeros((), dtype=torch.float32)
            for start in range(0, len(labels), 256):
                h, y = (flat[start:start + 256], labels[start:start + 256])
                loss = loss + (checkpoint(chunk_loss, h, y, use_reentrant=False) if self.training else chunk_loss(h, y))
            return loss
        return (F.linear(x[:, -1], self.token.weight), kvs)

class ResidualAdapter(nn.Module):
    def __init__(self,w,r):
        super().__init__(); self.down=nn.Linear(w,r); self.up=nn.Linear(r,w)
        nn.init.normal_(self.down.weight,std=.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self,x): return x+self.up(F.gelu(self.down(x)))

class LoRMAPlus(nn.Module):
    def __init__(self,linear,rank,alpha,groups=1):
        super().__init__(); self.weight=linear.weight; self.bias=linear.bias
        self.weight.requires_grad_(False)
        if self.bias is not None: self.bias.requires_grad_(False)
        self.groups=groups; self.width=linear.out_features//groups; self.scale=alpha/rank
        assert linear.out_features%groups==0 and 0<rank<=self.width
        self.lorma_A=nn.Parameter(torch.empty(groups,rank,self.width))
        self.lorma_B=nn.Parameter(torch.zeros(groups,self.width,rank))
        nn.init.normal_(self.lorma_A,std=.02)
    def forward(self,x):
        z=F.linear(x,self.weight,None)
        chunks=z.split(self.width,dim=-1)
        result=torch.cat([h+self.scale*F.linear(F.linear(h,self.lorma_A[i]),self.lorma_B[i])
                          for i,h in enumerate(chunks)],dim=-1)
        return result if self.bias is None else result+self.bias

class Model1Backbone(DecoderLM):
    def __init__(self,model_cfg,cfg):
        super().__init__(**model_cfg); self.method=cfg['method']
        self.gradient_checkpointing=cfg['activation_checkpointing']
        if self.method!='full':
            for p in self.parameters(): p.requires_grad_(False)
        if self.method=='adapters':
            self.attention_adapters=nn.ModuleList([ResidualAdapter(model_cfg['width'],cfg['adapter_dim']) for _ in self.blocks])
            self.ff_adapters=nn.ModuleList([ResidualAdapter(model_cfg['width'],cfg['adapter_dim']) for _ in self.blocks])
        elif self.method=='lorma':
            for block in self.blocks:
                block.attn.qkv=LoRMAPlus(block.attn.qkv,cfg['lorma_rank'],cfg['lorma_alpha'],groups=3)
                block.attn.proj=LoRMAPlus(block.attn.proj,cfg['lorma_rank'],cfg['lorma_alpha'])
        elif self.method!='full': raise ValueError('Unknown method')
    def load_base(self,state):
        extra={k for k in self.state_dict() if k.startswith(('attention_adapters.','ff_adapters.')) or '.lorma_' in k}
        if set(state)!=set(self.state_dict())-extra: raise ValueError('Model-1 checkpoint keys do not match')
        result=self.load_state_dict(state,strict=False)
        if set(result.missing_keys)!=extra or result.unexpected_keys: raise ValueError('Incomplete pretrained weights')
    def hidden(self,ids):
        if ids.shape[1]>self.config['context']: raise ValueError('Context exceeded')
        x=self.drop(self.token(ids)+self.position(torch.arange(ids.shape[1],device=ids.device)))
        for i,block in enumerate(self.blocks):
            def layer(z,block=block,i=i):
                if self.method!='adapters': return block(z)[0]
                z=self.attention_adapters[i](z+block.drop(block.attn(block.ln1(z))[0]))
                return self.ff_adapters[i](z+block.drop(block.mlp(block.ln2(z))))
            x=checkpoint(layer,x,use_reentrant=False) if self.training and self.gradient_checkpointing else layer(x)
        return self.final_norm(x),None


In [ ]:
class TeacherNER(nn.Module):
    def __init__(self,model_cfg,nlabels,cfg,weights):
        super().__init__()
        self.backbone=Model1Backbone(model_cfg,cfg)
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(cfg['seed']+1000)
            w=model_cfg['width']; d=cfg['head_dim']
            self.bi=nn.TransformerEncoderLayer(
                d_model=w,nhead=model_cfg['heads'],dim_feedforward=2*w,
                dropout=cfg['bi_dropout'],batch_first=True,norm_first=True)
            self.projection=nn.Linear(w,d)
            self.characters=nn.Embedding(cfg['char_buckets'],d)
            self.positions=nn.Embedding(256,d)
            self.mix=nn.Sequential(nn.Conv1d(d,d,9,padding=4),nn.GELU(),nn.Conv1d(d,d,9,padding=4))
            self.head=nn.Sequential(nn.LayerNorm(d),nn.GELU(),nn.Linear(d,nlabels))
            self.register_buffer('class_weights',torch.as_tensor(weights,dtype=torch.float32))

    def forward(self,x,mapping,chars,positions,y=None):
        valid=mapping.ge(0)
        h,_=self.backbone.hidden(x)
        h=self.bi(h,src_key_padding_mask=x.eq(PAD))
        h=self.projection(h)
        h=h.gather(1,mapping.clamp_min(0)[:,:,None].expand(-1,-1,h.shape[-1]))
        mask=valid[:,:,None].to(h.dtype)
        z=(h+self.characters(chars)+self.positions(positions))*mask
        conv_mask=mask.transpose(1,2)
        mixed=F.gelu(self.mix[0](z.transpose(1,2)))*conv_mask
        mixed=self.mix[2](mixed)*conv_mask
        logits=self.head((z+mixed.transpose(1,2))*mask)
        if y is None: return logits
        if not torch.equal(valid,y.ne(-100)): raise ValueError('Character padding mask mismatch')
        return F.cross_entropy(logits.float().flatten(0,1),y.flatten(),
            weight=self.class_weights,ignore_index=-100,reduction='sum').reshape(1)


class CRFTeacher(TeacherNER):
    def __init__(self,model_cfg,nlabels,cfg,weights):
        super().__init__(model_cfg,nlabels,cfg,weights)
        self.crf_transitions=nn.Parameter(torch.zeros(nlabels,nlabels))


In [ ]:
class CausalGatedConv(nn.Module):
    def __init__(self,width,kernel_size,dilation):
        super().__init__()
        if kernel_size<2 or dilation<1: raise ValueError('Invalid causal convolution settings')
        self.history=(kernel_size-1)*dilation
        self.conv=nn.Conv1d(width,2*width,kernel_size,dilation=dilation,padding=0)
        self.proj=nn.Linear(width,width)

    def forward(self,x,past=None,cache=False):
        values=x.transpose(1,2)
        if past is None:
            padded=F.pad(values,(self.history,0))
        else:
            if past.shape!=(x.shape[0],x.shape[2],self.history):
                raise ValueError('Convolution cache shape mismatch')
            padded=torch.cat((past,values),dim=2)
        features=self.conv(padded)
        a,b=features.chunk(2,dim=1)
        output=self.proj((a*torch.sigmoid(b)).transpose(1,2))
        # clone prevents a small cache view from retaining a full prompt buffer.
        state=padded[:,:,-self.history:].clone() if cache else None
        return output,state

class ConvBlock(nn.Module):
    def __init__(self,width,kernel_size,dilation,dropout):
        super().__init__()
        self.n1=nn.LayerNorm(width); self.n2=nn.LayerNorm(width)
        self.causal=CausalGatedConv(width,kernel_size,dilation)
        self.ff=nn.Sequential(nn.Linear(width,4*width),nn.GELU(),nn.Linear(4*width,width))
        self.drop=nn.Dropout(dropout)

    def forward(self,x,past=None,cache=False):
        h,state=self.causal(self.n1(x),past,cache)
        x=x+self.drop(h)
        return x+self.drop(self.ff(self.n2(x))),state


In [ ]:
class GatedConvLM(nn.Module):
    def __init__(self,vocab_size=32768,width=640,layers=16,kernel_size=3,dilation_cycle=10,
                 context=2048,dropout=0.0,activation_checkpointing=True,loss_chunk=256):
        super().__init__()
        if min(width,layers,dilation_cycle,context,loss_chunk)<1 or kernel_size<2:
            raise ValueError('Invalid model configuration')
        self.config=dict(vocab_size=vocab_size,width=width,layers=layers,kernel_size=kernel_size,
            dilation_cycle=dilation_cycle,context=context,dropout=dropout,
            activation_checkpointing=activation_checkpointing,loss_chunk=loss_chunk)
        self.dilations=[2**(i%dilation_cycle) for i in range(layers)]
        self.receptive_field=1+(kernel_size-1)*sum(self.dilations)
        self.token=nn.Embedding(vocab_size,width)
        self.blocks=nn.ModuleList([ConvBlock(width,kernel_size,d,dropout) for d in self.dilations])
        self.norm=nn.LayerNorm(width); self.drop=nn.Dropout(dropout)
        self.apply(self.initialize)
        for block in self.blocks:
            nn.init.normal_(block.causal.proj.weight,std=0.02/math.sqrt(2*layers))
            nn.init.normal_(block.ff[2].weight,std=0.02/math.sqrt(2*layers))

    @staticmethod
    def initialize(module):
        if isinstance(module,(nn.Linear,nn.Conv1d,nn.Embedding)):
            nn.init.normal_(module.weight,std=0.02)
            if getattr(module,'bias',None) is not None: nn.init.zeros_(module.bias)

    def hidden(self,ids,past=None,cache=False):
        if past is None and ids.shape[1]>self.config['context']:
            raise ValueError('Initial prompt/training block exceeds configured context')
        if past is not None and (len(past)!=len(self.blocks) or ids.shape[1]!=1):
            raise ValueError('Cached generation accepts one token and one state per layer')
        if self.training and (cache or past is not None):
            raise ValueError('Caches are for eval-mode generation only')
        x=self.drop(self.token(ids)); states=[]
        for i,block in enumerate(self.blocks):
            if self.training and self.config['activation_checkpointing']:
                x=checkpoint(lambda z,layer=block: layer(z)[0],x,use_reentrant=False)
            else:
                x,state=block(x,None if past is None else past[i],cache)
                if cache: states.append(state)
        return self.norm(x),states

    def forward(self,ids,targets=None,past=None,cache=False):
        h,states=self.hidden(ids,past,cache)
        if targets is None: return F.linear(h[:,-1],self.token.weight),states
        flat=h.reshape(-1,h.shape[-1]); labels=targets.reshape(-1)
        def chunk_loss(a,b):
            return F.cross_entropy(F.linear(a,self.token.weight).float(),b,ignore_index=-100,reduction='sum')
        loss=h.new_zeros((),dtype=torch.float32); size=self.config['loss_chunk']
        for start in range(0,len(labels),size):
            a,b=flat[start:start+size],labels[start:start+size]
            loss=loss+(checkpoint(chunk_loss,a,b,use_reentrant=False) if self.training and self.config['activation_checkpointing'] else chunk_loss(a,b))
        return loss


In [ ]:
TEACHER_STATE=torch.load(TEACHER_DIR/'best.pt',map_location='cpu',weights_only=False)
TEACHER_SIGNATURE=TEACHER_STATE['signature']
if TEACHER_SIGNATURE['config'].get('head_version')!='model1-adapters-crf-v1': raise ValueError('Expected adapter + CRF teacher')
if read_json(TEACHER_DIR/'config.json')!=TEACHER_SIGNATURE: raise ValueError('Teacher config/checkpoint mismatch')
TOKENIZER=ScratchByteBPE.from_file(MODEL4_DIR/'tokenizer.json')
TEACHER_TOKENIZER=ScratchByteBPE.from_file(TEACHER_DIR/'tokenizer.json')
if digest_file(TEACHER_DIR/'tokenizer.json')!=TEACHER_SIGNATURE['tokenizer_sha256']: raise ValueError('Teacher tokenizer mismatch')
BASE_STATE=torch.load(BASE_CHECKPOINT,map_location='cpu',weights_only=False)
PARENT_MODEL_CONFIG=copy.deepcopy(BASE_STATE['model_config'])
if 'kernel_size' not in PARENT_MODEL_CONFIG: raise ValueError('Expected model-4 gated convolution checkpoint')
parent_hash=BASE_STATE['tokenizer_sha256']
if parent_hash!=digest_file(MODEL4_DIR/'tokenizer.json'): raise ValueError('Model-4 tokenizer mismatch')
if TOKENIZER.get_vocab_size()!=PARENT_MODEL_CONFIG['vocab_size']: raise ValueError('Model-4 vocabulary mismatch')
if not 0<CONFIG['student_layers']<PARENT_MODEL_CONFIG['layers']: raise ValueError('Student must have fewer blocks')
MODEL_CONFIG=dict(PARENT_MODEL_CONFIG,layers=CONFIG['student_layers'],heads=CONFIG['student_heads'])
assert MODEL_CONFIG['width']%MODEL_CONFIG['heads']==0
assert CONFIG['window']<=MODEL_CONFIG['context']
BASE_EXPOSURE=BASE_STATE.get('tokens',BASE_STATE.get('total_tokens'))
teacher_cfg=TEACHER_SIGNATURE['config']
for key in ('window','overlap','char_buckets'):
    if CONFIG[key]!=teacher_cfg[key]: raise ValueError(f'Keep teacher preparation setting: {key}')
print('Student:',MODEL_CONFIG)
print('Teacher best validation:',TEACHER_STATE['metrics']['macro_f1'])


Student: {'vocab_size': 32768, 'width': 640, 'layers': 8, 'kernel_size': 3, 'dilation_cycle': 10, 'context': 2048, 'dropout': 0.0, 'activation_checkpointing': False, 'loss_chunk': 256, 'heads': 8}
Teacher best validation: 0.5492515376309127


## 3. Exact teacher split and character alignment


In [ ]:
def strict_json(path):
    def pairs(items):
        out={}
        for k,v in items:
            if k in out: raise ValueError(f'Duplicate JSON key: {k}')
            out[k]=v
        return out
    return json.loads(Path(path).read_text(encoding='utf-8'),object_pairs_hook=pairs)

def load_corpus(root):
    labels=[x.strip() for x in (root/'Train/labels.txt').read_text(encoding='utf-8-sig').splitlines() if x.strip()]
    if len(labels)!=12 or len(set(labels))!=12: raise ValueError('Expected 12 distinct labels')
    texts={}; gold={}
    for fold in (1,2):
        t=strict_json(root/f'Train/fold{fold}-text.json'); g=strict_json(root/f'Train/fold{fold}-gold.json')
        if not isinstance(t,dict) or not isinstance(g,dict) or set(t)!=set(g): raise ValueError('Training ID mismatch')
        for key,text in t.items():
            if not isinstance(text,str) or not text: raise ValueError(f'Invalid text: {key}')
            spans=[]
            if not isinstance(g[key],list): raise ValueError(f'Expected entity list: {key}')
            for item in g[key]:
                if not isinstance(item,list) or len(item) not in (3,4): raise ValueError(f'Invalid entity: {key}: {item}')
                a,b,label=item[:3]
                if type(a)!=int or type(b)!=int or not 0<=a<b<=len(text) or label not in labels:
                    raise ValueError(f'Invalid span: {key}: {item}')
                if len(item)==4 and text[a:b]!=item[3]: raise ValueError(f'Offset/text mismatch: {key}: {item}')
                spans.append([a,b,label])
            spans.sort()
            if len({tuple(s) for s in spans})!=len(spans): raise ValueError(f'Duplicate annotations: {key}')
            if any(left[1]>right[0] for left,right in zip(spans,spans[1:])):
                raise ValueError(f'Overlapping gold spans in {key}; use an overlapping-span head before training')
            if key in texts and (texts[key]!=text or gold[key]!=spans): raise ValueError(f'Conflicting fold ID: {key}')
            texts[key]=text; gold[key]=spans
    test=strict_json(root/'Test/fold3-text.json')
    if not isinstance(test,dict) or not test or any(not isinstance(t,str) for t in test.values()): raise ValueError('Invalid test documents')
    return texts,gold,test,labels

def make_split(texts,gold,labels,seed,fraction):
    groups=defaultdict(list)
    for key,text in texts.items(): groups[hashlib.sha256(text.encode('utf-8')).hexdigest()].append(key)
    for keys in groups.values():
        if any(gold[k]!=gold[keys[0]] for k in keys): raise ValueError('Duplicate text with conflicting annotations')
    hashes=sorted(groups)
    if len(hashes)<2: raise ValueError('Need at least two distinct documents')
    count=max(1,min(len(hashes)-1,round(len(hashes)*fraction)))
    best=None
    for attempt in range(100):
        order=np.random.default_rng(seed+attempt).permutation(hashes)
        val=sorted(k for h in order[:count] for k in groups[h]); train=sorted(k for h in order[count:] for k in groups[h])
        coverage=[{s[2] for k in ids for s in gold[k]} for ids in (train,val)]
        score=(len(coverage[0]&coverage[1]),len(coverage[0]),len(coverage[1]))
        if best is None or score>best[0]: best=(score,train,val)
    return dict(train=best[1],val=best[2])

def char_targets(text,spans,labels):
    y=np.zeros(len(text),dtype=np.int64); lookup={l:i for i,l in enumerate(labels)}
    for a,b,label in spans:
        k=lookup[label]; y[a]=1+2*k; y[a+1:b]=2+2*k
    return y

def decode_tags(tags,labels):
    spans=[]; start=None; active=None
    for i,value in enumerate(list(tags)+[0]):
        value=int(value); kind=None if value==0 else (value-1)//2
        begins=value>0 and value%2==1
        if start is not None and (kind!=active or begins):
            spans.append([start,i,labels[active]]); start=None
        if kind is not None and start is None: start=i
        active=kind
    return spans


In [ ]:
def encode_document(text,tok):
    ids=np.asarray(tok.encode(text).ids,dtype=np.int64)
    byte_lengths=np.asarray([len(tok.token_bytes[int(i)]) for i in ids],dtype=np.int64)
    token_ends=np.cumsum(byte_lengths)
    char_ends=np.cumsum([len(ch.encode('utf-8')) for ch in text],dtype=np.int64)
    if not np.array_equal(np.asarray([sum(byte_lengths)]),np.asarray([len(text.encode('utf-8'))])): raise ValueError('Byte alignment failed')
    if tok.decode(ids)!=text: raise ValueError('Tokenizer round trip failed')
    mapping=np.searchsorted(token_ends,char_ends,side='left')
    positions=np.zeros(len(text),dtype=np.int64)
    first=0
    for i in range(len(text)):
        if i==0 or mapping[i]!=mapping[i-1]: first=i
        positions[i]=min(255,i-first)
    return dict(ids=ids,mapping=mapping,characters=np.asarray([ord(c) for c in text],dtype=np.int64),positions=positions)

class WindowData:
    def __init__(self,texts,keys,tok,cfg,gold=None,labels=None):
        self.docs={}; self.rows=[]; self.cfg=cfg
        for key in keys:
            text=texts[key]; doc=encode_document(text,tok)
            doc['targets']=char_targets(text,gold[key],labels) if gold is not None else np.zeros(len(text),dtype=np.int64)
            self.docs[key]=doc
            previous=0; start=0
            while start<len(doc['ids']):
                end=min(start+cfg['window'],len(doc['ids']))
                a=int(np.searchsorted(doc['mapping'],previous,'left'))
                b=int(np.searchsorted(doc['mapping'],end,'left'))
                if b>a: self.rows.append((key,start,end,a,b))
                previous=end
                if end==len(doc['ids']): break
                start=end-cfg['overlap']
    def __len__(self): return len(self.rows)
    def batch(self,indices,device):
        rows=[self.rows[int(i)] for i in indices]
        nt=max(r[2]-r[1] for r in rows); nc=max(r[4]-r[3] for r in rows)
        x=np.full((len(rows),nt),PAD,dtype=np.int64); mapping=np.full((len(rows),nc),-1,dtype=np.int64)
        chars=np.zeros_like(mapping); positions=np.zeros_like(mapping); y=np.full_like(mapping,-100)
        for i,(key,start,end,a,b) in enumerate(rows):
            doc=self.docs[key]; n=b-a
            x[i,:end-start]=doc['ids'][start:end]
            mapping[i,:n]=doc['mapping'][a:b]-start
            chars[i,:n]=doc['characters'][a:b]%self.cfg['char_buckets']
            positions[i,:n]=doc['positions'][a:b]; y[i,:n]=doc['targets'][a:b]
            assert np.all(mapping[i,:n]>=0)
        return tuple(torch.from_numpy(v).to(device) for v in (x,mapping,chars,positions,y))

def verify_windows(data,texts):
    covered=defaultdict(int)
    for key,start,end,a,b in data.rows:
        assert a==covered[key] and 0<=start<end<=len(data.docs[key]['ids'])
        covered[key]=b
    assert all(covered[k]==len(texts[k]) for k in data.docs)

def make_weights(data,nlabels,cap):
    counts=np.zeros(nlabels,dtype=np.int64)
    for doc in data.docs.values(): counts+=np.bincount(doc['targets'],minlength=nlabels)
    weights=np.ones(nlabels,dtype=np.float32)
    observed=counts>0
    weights[observed]=np.minimum(cap,np.sqrt(counts.max()/counts[observed]))
    weights/=np.sum(weights*counts)/max(1,counts.sum())
    return weights,counts


In [ ]:
TEXTS,GOLD,TEST_TEXTS,LABELS=load_corpus(DATA_ROOT)
assert LABELS==TEACHER_SIGNATURE['labels']
split=TEACHER_SIGNATURE['split']
assert read_json(TEACHER_DIR/'split.json')==split
assert set(split['train']).isdisjoint(split['val'])
assert set(split['train'])|set(split['val'])==set(TEXTS)
for name,digest in TEACHER_SIGNATURE['data'].items():
    if digest_file(DATA_ROOT/name)!=digest: raise ValueError(f'Teacher dataset mismatch: {name}; attach the original dataset')
if digest_file(EVALUATOR_PATH)!=TEACHER_SIGNATURE['evaluator_sha256']: raise ValueError('Evaluator differs from teacher')
CLASS_WEIGHTS=torch.ones(1+2*len(LABELS))
PLAIN_TRAIN=WindowData(TEXTS,split['train'],TOKENIZER,CONFIG,GOLD,LABELS)
VAL_DATA=WindowData(TEXTS,split['val'],TOKENIZER,CONFIG,GOLD,LABELS)
verify_windows(PLAIN_TRAIN,TEXTS); verify_windows(VAL_DATA,TEXTS)
print('Documents:',len(split['train']),'train,',len(split['val']),'validation')
print('Windows:',len(PLAIN_TRAIN),'train,',len(VAL_DATA),'validation')


In [ ]:
def validate_predictions(predictions,texts,labels):
    if set(predictions)!=set(texts): raise ValueError('Prediction IDs do not match')
    for key,spans in predictions.items():
        if not isinstance(spans,list): raise ValueError(f'Invalid prediction: {key}')
        seen=set()
        for item in spans:
            if len(item)!=3: raise ValueError('Expected start, end, label')
            a,b,label=item
            if type(a)!=int or type(b)!=int or not 0<=a<b<=len(texts[key]) or label not in labels:
                raise ValueError(f'Invalid span: {key}: {item}')
            if tuple(item) in seen: raise ValueError(f'Duplicate span: {key}')
            seen.add(tuple(item))

def span_metrics(gold,predictions,labels):
    if set(gold)!=set(predictions): raise ValueError('Evaluation ID mismatch')
    counts={label:[0,0,0] for label in labels}
    for key in gold:
        g={tuple(s[:3]) for s in gold[key]}; p={tuple(s[:3]) for s in predictions[key]}
        for item in g&p: counts[item[2]][0]+=1
        for item in p-g: counts[item[2]][1]+=1
        for item in g-p: counts[item[2]][2]+=1
    scores={}
    for label,(tp,fp,fn) in counts.items():
        precision=tp/(tp+fp) if tp+fp else 0.; recall=tp/(tp+fn) if tp+fn else 0.
        scores[label]=dict(precision=precision,recall=recall,f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.,tp=tp,fp=fp,fn=fn)
    return dict(macro_f1=float(np.mean([r['f1'] for r in scores.values()])),per_label=scores)

def viterbi_bio(logp):
    length,k=logp.shape
    if length==0: return np.empty(0,dtype=np.int64)
    if k%2!=1 or not np.isfinite(logp).all(): raise ValueError('Invalid BIO emissions')
    previous=np.arange(k)[:,None]; following=np.arange(k)[None,:]
    allowed=(following==0)|(following%2==1)|((previous>0)&((previous-1)//2==(following-1)//2))
    transition=np.where(allowed,0.,-np.inf).astype(np.float32)
    score=logp[0].copy(); score[2::2]=-np.inf
    back=np.zeros((length,k),dtype=np.int16)
    for t in range(1,length):
        candidates=score[:,None]+transition
        back[t]=candidates.argmax(axis=0)
        score=candidates.max(axis=0)+logp[t]
    path=np.zeros(length,dtype=np.int64); path[-1]=score.argmax()
    for t in range(length-1,0,-1): path[t-1]=back[t,path[t]]
    return path


## 4. Cache teacher probabilities

In [ ]:
def bio_allowed(k):
    a=np.arange(k)[:,None]; b=np.arange(k)[None,:]
    return (b==0)|(b%2==1)|((a>0)&((a-1)//2==(b-1)//2))

def stable_lse(x,axis):
    m=x.max(axis=axis,keepdims=True)
    return np.squeeze(m+np.log(np.exp(x-m).sum(axis=axis,keepdims=True)),axis=axis)

def teacher_marginals(emissions,transitions,temperature):
    e=np.asarray(emissions,dtype=np.float64)/temperature
    tr=np.where(bio_allowed(e.shape[1]),np.asarray(transitions,dtype=np.float64)/temperature,-np.inf)
    length,k=e.shape
    if not length: return np.zeros_like(e,dtype=np.float32)
    alpha=np.empty_like(e); scales=np.empty(length)
    first=e[0].copy(); first[2::2]=-np.inf
    scales[0]=stable_lse(first,0); alpha[0]=first-scales[0]
    for t in range(1,length):
        v=e[t]+stable_lse(alpha[t-1,:,None]+tr,0)
        scales[t]=stable_lse(v,0); alpha[t]=v-scales[t]
    probabilities=np.empty_like(e); beta=np.zeros(k)
    probabilities[-1]=np.exp(alpha[-1])
    for t in range(length-1,0,-1):
        beta=stable_lse(tr+e[t][None,:]+beta[None,:]-scales[t],1)
        probabilities[t-1]=np.exp(alpha[t-1]+beta)
    probabilities/=probabilities.sum(-1,keepdims=True)
    if not np.isfinite(probabilities).all(): raise RuntimeError('Invalid teacher probabilities')
    return probabilities.astype(np.float32)

def learned_decode(emissions,transitions):
    length,k=emissions.shape
    if length==0: return np.empty(0,dtype=np.int64)
    tr=np.where(bio_allowed(k),transitions,-np.inf)
    score=emissions[0].astype(np.float64).copy(); score[2::2]=-np.inf
    back=np.zeros((length,k),dtype=np.int16)
    for t in range(1,length):
        candidates=score[:,None]+tr
        back[t]=candidates.argmax(0); score=candidates.max(0)+emissions[t]; score-=score.max()
    path=np.empty(length,dtype=np.int64); path[-1]=score.argmax()
    for t in range(length-1,0,-1): path[t-1]=back[t,path[t]]
    return path

@torch.inference_mode()
def document_emissions(model,data,device):
    values={key:np.zeros((len(doc['mapping']),1+2*len(LABELS)),dtype=np.float32) for key,doc in data.docs.items()}
    for i,(key,_,_,a,b) in enumerate(data.rows):
        args=data.batch([i],device)
        with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=device.type=='cuda'):
            logits=model(*args[:-1])
        if not torch.isfinite(logits).all(): raise RuntimeError('Non-finite emissions')
        values[key][a:b]=logits[0,:b-a].float().cpu().numpy()
    return values

@torch.inference_mode()
def predict_windows(model,data,labels,device):
    was_training=model.training; model.eval()
    try:
        transitions=model.crf_transitions.detach().float().cpu().numpy()
        return {key:decode_tags(learned_decode(e,transitions),labels)
                for key,e in document_emissions(model,data,device).items()}
    finally: model.train(was_training)

def cache_teacher():
    global CACHE_DIR
    CACHE_DIR=ASSET_DIR/'teacher_cache'
    CACHE_DIR.mkdir(parents=True,exist_ok=True)
    signature=dict(version=1,teacher=digest_file(TEACHER_DIR/'best.pt'),
        teacher_tokenizer=digest_file(TEACHER_DIR/'tokenizer.json'),
        data=TEACHER_SIGNATURE['data'],split=split,temperature=CONFIG['kd_temperature'],labels=LABELS)
    manifest_path=CACHE_DIR/'manifest.json'
    if manifest_path.exists():
        manifest=read_json(manifest_path)
        if manifest['signature']!=signature: raise ValueError('Teacher cache settings changed; use a new output root')
    else:
        if any(CACHE_DIR.iterdir()): raise ValueError('Cache has files without metadata')
        manifest=dict(signature=signature,files={})
        write_json(manifest_path,manifest)
    for key,info in manifest['files'].items():
        if key not in split['train']: raise ValueError('Non-training document in cache')
        if digest_file(CACHE_DIR/info['file'])!=info['sha256']: raise ValueError('Teacher cache file changed')
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model=CRFTeacher(TEACHER_SIGNATURE['model'],1+2*len(LABELS),TEACHER_SIGNATURE['config'],TEACHER_STATE['model']['class_weights'])
    model.load_state_dict(TEACHER_STATE['model'],strict=True); model.to(device).eval()
    for p in model.parameters(): p.requires_grad_(False)
    try:
        validation=WindowData(TEXTS,split['val'],TEACHER_TOKENIZER,TEACHER_SIGNATURE['config'])
        print('Checking teacher validation predictions...',flush=True)
        predictions=predict_windows(model,validation,LABELS,device)
        metrics=span_metrics({k:GOLD[k] for k in split['val']},predictions,LABELS)
        write_json(ASSET_DIR/'teacher_validation_check.json',metrics)
        expected=TEACHER_STATE['metrics']['macro_f1']
        print('Teacher expected:',expected,'reproduced:',metrics['macro_f1'],flush=True)
        if abs(metrics['macro_f1']-expected)>1e-6:
            raise ValueError('Teacher score did not reproduce; stop before distillation and check model/data/precision')
        transitions=model.crf_transitions.detach().float().cpu().numpy()
        for index,key in enumerate(split['train']):
            if key not in manifest['files']:
                data=WindowData(TEXTS,[key],TEACHER_TOKENIZER,TEACHER_SIGNATURE['config'])
                logits=document_emissions(model,data,device)[key]
                probabilities=teacher_marginals(logits,transitions,CONFIG['kd_temperature'])
                name=hashlib.sha256(key.encode()).hexdigest()+'.npy'
                path=CACHE_DIR/name; temp=path.with_suffix('.tmp')
                with temp.open('wb') as f: np.save(f,probabilities.astype(np.float16),allow_pickle=False)
                os.replace(temp,path)
                manifest['files'][key]=dict(file=name,sha256=digest_file(path),characters=len(TEXTS[key]))
                write_json(manifest_path,manifest)
            print(f"Teacher cache {index+1}/{len(split['train'])}",flush=True)
    finally:
        del model
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

class DistillationData(WindowData):
    def __init__(self,*args,**kwargs):
        super().__init__(*args,**kwargs)
        manifest=read_json(CACHE_DIR/'manifest.json')
        if set(self.docs)!=set(manifest['files']): raise ValueError('Cache must contain exactly the training split')
        self.soft={key:np.load(CACHE_DIR/info['file'],mmap_mode='r',allow_pickle=False) for key,info in manifest['files'].items()}
        for key,array in self.soft.items():
            if array.shape!=(len(self.docs[key]['mapping']),1+2*len(LABELS)): raise ValueError('Teacher target shape mismatch')
    def batch(self,indices,device):
        x,m,c,p,y=super().batch(indices,device)
        soft=np.zeros((*m.shape,1+2*len(LABELS)),dtype=np.float32)
        for row,index in enumerate(indices):
            key,_,_,a,b=self.rows[int(index)]
            soft[row,:b-a]=self.soft[key][a:b]
        return x,m,c,p,torch.from_numpy(soft).to(device),y


In [ ]:
cache_teacher()
TRAIN_DATA=DistillationData(TEXTS,split['train'],TOKENIZER,CONFIG,GOLD,LABELS)
print('Cached training probabilities ready.')


## 5. Student and distillation loss


In [ ]:
class LegalNER(nn.Module):
    def __init__(self,model_cfg,nlabels,cfg,weights):
        super().__init__()
        self.kd_temperature=cfg['kd_temperature']; self.kd_weight=cfg['kd_weight']
        self.backbone=GatedConvLM(**{k:v for k,v in model_cfg.items() if k!='heads'})
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(cfg['seed']+1000)
            w=model_cfg['width']; d=cfg['head_dim']
            self.bi=nn.TransformerEncoderLayer(
                d_model=w,nhead=model_cfg['heads'],dim_feedforward=2*w,
                dropout=cfg['bi_dropout'],batch_first=True,norm_first=True)
            self.projection=nn.Linear(w,d)
            self.characters=nn.Embedding(cfg['char_buckets'],d)
            self.positions=nn.Embedding(256,d)
            self.mix=nn.Sequential(nn.Conv1d(d,d,9,padding=4),nn.GELU(),nn.Conv1d(d,d,9,padding=4))
            self.head=nn.Sequential(nn.LayerNorm(d),nn.GELU(),nn.Linear(d,nlabels))
            self.register_buffer('class_weights',torch.as_tensor(weights,dtype=torch.float32))

        self.register_buffer('crf_transitions',TEACHER_TRANSITIONS.clone())

    def forward(self,x,mapping,chars,positions,soft=None,y=None):
        valid=mapping.ge(0)
        h,_=self.backbone.hidden(x)
        h=self.bi(h,src_key_padding_mask=x.eq(PAD))
        h=self.projection(h)
        h=h.gather(1,mapping.clamp_min(0)[:,:,None].expand(-1,-1,h.shape[-1]))
        mask=valid[:,:,None].to(h.dtype)
        z=(h+self.characters(chars)+self.positions(positions))*mask
        conv_mask=mask.transpose(1,2)
        mixed=F.gelu(self.mix[0](z.transpose(1,2)))*conv_mask
        mixed=self.mix[2](mixed)*conv_mask
        logits=self.head((z+mixed.transpose(1,2))*mask)
        if y is None: return logits
        if not torch.equal(valid,y.ne(-100)): raise ValueError('Character padding mask mismatch')
        if soft is None: raise ValueError('Missing teacher targets')
        ce=F.cross_entropy(logits.float().flatten(0,1),y.flatten(),ignore_index=-100,reduction='sum')
        temperature=self.kd_temperature
        logp=F.log_softmax(logits.float()/temperature,dim=-1)
        probs=soft.float().clamp_min(0)
        probs=probs/probs.sum(-1,keepdim=True).clamp_min(1e-12)
        kl=(probs*(probs.clamp_min(1e-12).log()-logp)).sum(-1)
        kd=kl.masked_select(valid).sum()*temperature**2
        return ((1-self.kd_weight)*ce+self.kd_weight*kd).reshape(1)

def make_model(cfg,device):
    mc=copy.deepcopy(MODEL_CONFIG); mc['activation_checkpointing']=cfg['activation_checkpointing']
    model=LegalNER(mc,1+2*len(LABELS),cfg,CLASS_WEIGHTS)
    expected=model.backbone.state_dict()
    retained={k:BASE_STATE['model'][k] for k in expected}
    model.backbone.load_state_dict(retained,strict=True)
    return model.to(device)

class PrecisionWrapper(nn.Module):
    def __init__(self,model): super().__init__(); self.model=model
    def forward(self,*args):
        with torch.autocast(device_type=args[0].device.type,dtype=torch.float16,enabled=args[0].is_cuda):
            return self.model(*args)

def setup_training(cfg,cpu=False):
    device=torch.device('cpu' if cpu else 'cuda')
    model=make_model(cfg,device)
    groups=[dict(params=[p for p in model.backbone.parameters() if p.requires_grad],lr=cfg['backbone_lr'],base_lr=cfg['backbone_lr']),
            dict(params=[p for n,p in model.named_parameters() if not n.startswith('backbone.')],lr=cfg['head_lr'],base_lr=cfg['head_lr'])]
    optimizer=torch.optim.AdamW(groups,weight_decay=cfg['weight_decay'])
    scaler=torch.amp.GradScaler('cuda',enabled=not cpu)
    wrapper=PrecisionWrapper(model)
    if not cpu and torch.cuda.device_count()>1: wrapper=nn.DataParallel(wrapper)
    return model,optimizer,scaler,wrapper,device

def update_batch(model,optimizer,scaler,wrapper,data,indices,cfg,device):
    batch=cfg['micro_batch']*(torch.cuda.device_count() if device.type=='cuda' else 1)
    denominator=sum(float(CLASS_WEIGHTS[data.docs[key]['targets'][a:b]].sum())
                    for key,_,_,a,b in (data.rows[int(i)] for i in indices))
    if denominator<=0: raise ValueError('Empty training batch')
    model.train()
    for retry in range(5):
        optimizer.zero_grad(set_to_none=True); loss_total=torch.zeros((),device=device)
        for start in range(0,len(indices),batch):
            args=data.batch(indices[start:start+batch],device)
            loss=wrapper(*args).sum(); loss_total+=loss.detach()
            scaler.scale(loss/denominator).backward()
        scaler.unscale_(optimizer)
        norm=torch.nn.utils.clip_grad_norm_(model.parameters(),cfg['clip'])
        if device.type=='cpu' and not torch.isfinite(norm): raise RuntimeError('Non-finite gradients')
        old=scaler.get_scale(); scaler.step(optimizer); scaler.update()
        if scaler.get_scale()>=old:
            value=float(loss_total)/denominator
            if not math.isfinite(value): raise RuntimeError('Non-finite loss')
            return value
    raise RuntimeError('Repeated FP16 overflow; checkpoint position unchanged')


In [ ]:
TEACHER_TRANSITIONS=TEACHER_STATE['model']['crf_transitions'].float().clone()
probe=make_model(CONFIG,'cpu')
teacher_params=sum(v.numel() for k,v in TEACHER_STATE['model'].items() if k!='class_weights')
student_params=sum(p.numel() for p in probe.parameters())
assert student_params<teacher_params
report=dict(teacher_parameters=teacher_params,student_parameters=student_params,
    reduction_fraction=1-student_params/teacher_params,receptive_field=probe.backbone.receptive_field,
    retained_blocks=list(range(CONFIG['student_layers'])))
write_json(ASSET_DIR/'compression.json',report); print(report)
del probe; gc.collect()


{'teacher_parameters': 127643978, 'student_parameters': 74407833, 'reduction_fraction': 0.4170674232669245, 'receptive_field': 511, 'retained_blocks': [0, 1, 2, 3, 4, 5, 6, 7]}


## 6. GPU capacity and speed


In [ ]:
def sync_devices():
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()): torch.cuda.synchronize(i)

def gpu_sweep(cfg):
    if not torch.cuda.is_available(): raise RuntimeError('Enable GPU acceleration')
    results=[]; saved=rng_state()
    nt=max(end-start for data in (TRAIN_DATA,VAL_DATA) for _,start,end,_,_ in data.rows)
    nc=max(b-a for data in (TRAIN_DATA,VAL_DATA) for _,_,_,a,b in data.rows)
    gpus=torch.cuda.device_count()
    class StressData:
        def __init__(self):
            self.rows=[('stress',0,nt,0,nc)]*cfg['effective_batch']
            self.docs={'stress':dict(targets=np.zeros(nc,dtype=np.int64))}
        def batch(self,indices,device):
            b=len(indices)
            return (torch.randint(4,MODEL_CONFIG['vocab_size'],(b,nt),device=device),
                    torch.randint(0,nt,(b,nc),device=device),
                    torch.randint(0,cfg['char_buckets'],(b,nc),device=device),
                    torch.randint(0,256,(b,nc),device=device),
                    torch.full((b,nc,1+2*len(LABELS)),1/(1+2*len(LABELS)),device=device),torch.zeros((b,nc),dtype=torch.long,device=device))
    try:
        for ckpt in (True,False):
            for micro in (1,2,4):
                if cfg['effective_batch']%(micro*gpus): continue
                candidate=dict(cfg,micro_batch=micro,activation_checkpointing=ckpt)
                model=optimizer=scaler=wrapper=None
                gc.collect(); torch.cuda.empty_cache()
                other=[]
                for i in range(gpus):
                    with torch.cuda.device(i): free,total=torch.cuda.mem_get_info()
                    other.append(max(0,total-free-torch.cuda.memory_reserved(i)))
                    torch.cuda.reset_peak_memory_stats(i)
                row=dict(micro_batch=micro,activation_checkpointing=ckpt,passed=False)
                try:
                    seed_all(cfg['seed']); model,optimizer,scaler,wrapper,device=setup_training(candidate)
                    stress=StressData()
                    for _ in range(2): update_batch(model,optimizer,scaler,wrapper,stress,list(range(cfg['effective_batch'])),candidate,device)
                    indices=list(range(min(cfg['effective_batch'],len(TRAIN_DATA))))
                    update_batch(model,optimizer,scaler,wrapper,TRAIN_DATA,indices,candidate,device)
                    sync_devices(); started=time.monotonic()
                    for _ in range(3): update_batch(model,optimizer,scaler,wrapper,TRAIN_DATA,indices,candidate,device)
                    sync_devices(); seconds=(time.monotonic()-started)/3
                    memory=[]
                    for i in range(gpus):
                        total=torch.cuda.get_device_properties(i).total_memory; reserved=torch.cuda.max_memory_reserved(i)
                        memory.append(dict(gpu=i,reserved_gib=reserved/2**30,headroom_gib=(total-reserved-other[i])/2**30))
                    row.update(seconds_per_update=seconds,memory=memory,passed=all(m['headroom_gib']>=1.5 for m in memory))
                except torch.cuda.OutOfMemoryError: row['error']='out of memory'
                finally:
                    del model,optimizer,scaler,wrapper
                    gc.collect(); torch.cuda.empty_cache()
                results.append(row); print(row,flush=True)
                if 'error' in row: break
    finally: restore_rng(saved)
    safe=[r for r in results if r['passed']]
    if not safe: raise RuntimeError('No configuration passed; reduce the effective batch/window and rerun preparation')
    best=min(safe,key=lambda r:r['seconds_per_update'])
    write_json(ASSET_DIR/'hardware.json',dict(results=results,selected=best))
    return best


In [ ]:
if not (RUN_DIR/'config.json').exists():
    selected=gpu_sweep(CONFIG)
    CONFIG['micro_batch']=selected['micro_batch']
    CONFIG['activation_checkpointing']=selected['activation_checkpointing']
    print('Training-only epoch minutes:',math.ceil(len(TRAIN_DATA)/CONFIG['effective_batch'])*selected['seconds_per_update']/60)
else: print('Using saved batch settings')
print(CONFIG)


{'micro_batch': 1, 'activation_checkpointing': True, 'passed': True, 'seconds_per_update': 0.2993365629999971, 'memory': [{'gpu': 0, 'reserved_gib': 1.919921875, 'headroom_gib': 12.51055908203125}, {'gpu': 1, 'reserved_gib': 0.46484375, 'headroom_gib': 13.99493408203125}]}
{'micro_batch': 2, 'activation_checkpointing': True, 'passed': True, 'seconds_per_update': 0.2223420073333576, 'memory': [{'gpu': 0, 'reserved_gib': 1.654296875, 'headroom_gib': 12.68243408203125}, {'gpu': 1, 'reserved_gib': 0.474609375, 'headroom_gib': 13.87188720703125}]}
{'micro_batch': 1, 'activation_checkpointing': False, 'passed': True, 'seconds_per_update': 0.23683616833333568, 'memory': [{'gpu': 0, 'reserved_gib': 1.939453125, 'headroom_gib': 12.39532470703125}, {'gpu': 1, 'reserved_gib': 0.65625, 'headroom_gib': 13.69024658203125}]}
{'micro_batch': 2, 'activation_checkpointing': False, 'passed': True, 'seconds_per_update': 0.1960702256666688, 'memory': [{'gpu': 0, 'reserved_gib': 1.599609375, 'headroom_gib':

## 7. Training and resumable checkpoints


In [ ]:
def experiment_signature(cfg):
    paths=['Train/fold1-text.json','Train/fold1-gold.json','Train/fold2-text.json',
           'Train/fold2-gold.json','Train/labels.txt','Test/fold3-text.json']
    return dict(version='phase2-model4-crf-posterior-kd-v1',teacher_sha256=digest_file(TEACHER_DIR/'best.pt'),teacher_cache=read_json(CACHE_DIR/'manifest.json'),config=cfg,model=MODEL_CONFIG,
        parent_sha256=digest_file(BASE_CHECKPOINT),tokenizer_sha256=digest_file(BASE_RUN/'tokenizer.json'),
        parent_exposure=BASE_EXPOSURE,labels=LABELS,split=split,
        data={name:digest_file(DATA_ROOT/name) for name in paths},
        evaluator_sha256=digest_file(EVALUATOR_PATH),gpus=torch.cuda.device_count())

def save_training(s):
    atomic_save(dict(model=s.model.state_dict(),optimizer=s.optimizer.state_dict(),
        scaler=s.scaler.state_dict(),rng=rng_state(),signature=s.signature,
        epoch=s.epoch,position=s.position,step=s.step,best=s.best),s.out/'latest.pt')
    write_json(s.out/'progress.json',dict(epoch=s.epoch,position=s.position,step=s.step,
        best_macro_f1=s.best,complete=s.epoch>=s.cfg['epochs']))

def log_training(s,**row):
    record=dict(step=s.step,epoch=s.epoch,**row)
    with (s.out/'metrics.jsonl').open('a',encoding='utf-8') as handle:
        handle.write(json.dumps(record)+'\n')
    print(record,flush=True)

def evaluate_session(s):
    saved=rng_state()
    try:
        pred=predict_windows(s.model,VAL_DATA,LABELS,s.device)
        texts={k:TEXTS[k] for k in split['val']}; truth={k:GOLD[k] for k in split['val']}
        validate_predictions(pred,texts,LABELS)
        scores=span_metrics(truth,pred,LABELS)
        log_training(s,validation_macro_f1=scores['macro_f1'])
        if scores['macro_f1']>s.best:
            s.best=scores['macro_f1']
            atomic_save(dict(model=s.model.state_dict(),signature=s.signature,step=s.step,
                epoch=s.epoch,metrics=scores),s.out/'best.pt')
            write_json(s.out/'best_validation_predictions.json',pred)
            write_json(s.out/'best_validation_metrics.json',scores)
        return scores
    finally: restore_rng(saved)

def open_session(cfg,cpu=False):
    if not cpu and not torch.cuda.is_available(): raise RuntimeError('Enable GPUs')
    signature=experiment_signature(cfg)
    if cfg['effective_batch']%(max(1,signature['gpus'])*cfg['micro_batch']): raise ValueError('Effective batch mismatch')
    RUN_DIR.mkdir(parents=True,exist_ok=True)
    config_path=RUN_DIR/'config.json'
    if config_path.exists():
        if read_json(config_path)!=signature: raise ValueError('Run configuration changed; use a new RUN_DIR')
    elif any(RUN_DIR.iterdir()): raise ValueError('Nonempty output directory without config.json')
    write_json(config_path,signature); write_json(RUN_DIR/'split.json',split)
    shutil.copy2(BASE_RUN/'tokenizer.json',RUN_DIR/'tokenizer.json')
    shutil.copy2(EVALUATOR_PATH,RUN_DIR/'evaluate_phase2.py')
    shutil.copy2(DATA_ROOT/'Train/labels.txt',RUN_DIR/'labels.txt')
    write_json(RUN_DIR/'validation_truth.json',{k:GOLD[k] for k in split['val']})
    seed_all(cfg['seed'])
    model,optimizer,scaler,wrapper,device=setup_training(cfg,cpu)
    s=SimpleNamespace(model=model,optimizer=optimizer,scaler=scaler,wrapper=wrapper,device=device,
        out=RUN_DIR,cfg=cfg,signature=signature,epoch=0,position=0,step=0,best=-1.)
    path=RUN_DIR/'latest.pt'
    if path.exists():
        state=torch.load(path,map_location='cpu',weights_only=False)
        if state['signature']!=signature: raise ValueError('Checkpoint signature mismatch')
        model.load_state_dict(state['model']); optimizer.load_state_dict(state['optimizer']); scaler.load_state_dict(state['scaler'])
        for name in ('epoch','position','step','best'): setattr(s,name,state[name])
        restore_rng(state['rng'])
    return s

def schedule_ratio(step,total,cfg):
    warm=max(1,round(total*cfg['warmup_fraction']))
    if step<warm: return (step+1)/warm
    progress=min(1.,(step-warm)/max(1,total-warm))
    return cfg['min_lr_ratio']+(1-cfg['min_lr_ratio'])*0.5*(1+math.cos(math.pi*progress))

def train_run(cfg,additional_steps=None,cpu=False):
    s=open_session(cfg,cpu); started=time.monotonic(); initial=s.step
    total=math.ceil(len(TRAIN_DATA)/cfg['effective_batch'])*cfg['epochs']
    try:
        if s.best<0: evaluate_session(s); save_training(s)
        while s.epoch<cfg['epochs']:
            if additional_steps is not None and s.step-initial>=additional_steps: break
            if time.monotonic()-started>=cfg['max_hours']*3600: break
            order=np.random.default_rng(cfg['seed']+s.epoch).permutation(len(TRAIN_DATA))
            end=min(len(order),s.position+cfg['effective_batch'])
            indices=order[s.position:end]
            ratio=schedule_ratio(s.step,total,cfg)
            for group in s.optimizer.param_groups: group['lr']=group['base_lr']*ratio
            loss=update_batch(s.model,s.optimizer,s.scaler,s.wrapper,TRAIN_DATA,indices,cfg,s.device)
            s.position=end; s.step+=1
            if s.position==len(order): s.epoch+=1; s.position=0; epoch_end=True
            else: epoch_end=False
            if s.step%cfg['log_every']==0 or s.step==initial+1:
                log_training(s,loss=loss,backbone_lr=s.optimizer.param_groups[0]['lr'],
                    elapsed_minutes=(time.monotonic()-started)/60)
            if s.step%cfg['eval_every']==0 or epoch_end: evaluate_session(s); save_training(s)
            elif s.step%cfg['save_every']==0: save_training(s)
        evaluate_session(s); save_training(s)
        report=dict(steps=s.step-initial,seconds=time.monotonic()-started,
            best_macro_f1=s.best,complete=s.epoch>=cfg['epochs'])
        write_json(s.out/'session_report.json',report); print(report)
        return report
    finally:
        del s
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
short_report=train_run(CONFIG,additional_steps=30)


{'step': 0, 'epoch': 0, 'validation_macro_f1': 0.0}
{'step': 1, 'epoch': 0, 'loss': 7.43054484027401, 'backbone_lr': 1.567398119122257e-07, 'elapsed_minutes': 0.11776049853333462}
{'step': 20, 'epoch': 0, 'loss': 6.068291661023835, 'backbone_lr': 3.134796238244514e-06, 'elapsed_minutes': 0.1685387257333332}
{'step': 30, 'epoch': 0, 'validation_macro_f1': 0.0}


The 30-update check saves a checkpoint. The next cell resumes training under the same schedule.

In [ ]:
training_report=train_run(CONFIG)


{'step': 31, 'epoch': 0, 'loss': 4.517919790815731, 'backbone_lr': 4.8589341692789965e-06, 'elapsed_minutes': 0.0034595240333336126}
{'step': 40, 'epoch': 0, 'loss': 3.3278394625452954, 'backbone_lr': 6.269592476489028e-06, 'elapsed_minutes': 0.027589813450000143}
{'step': 60, 'epoch': 0, 'loss': 2.674850855922405, 'backbone_lr': 9.404388714733543e-06, 'elapsed_minutes': 0.08023222230000329}
{'step': 80, 'epoch': 0, 'loss': 2.4445779279921434, 'backbone_lr': 1.2539184952978056e-05, 'elapsed_minutes': 0.13337871401666918}
{'step': 100, 'epoch': 0, 'loss': 2.096749411517117, 'backbone_lr': 1.5673981191222574e-05, 'elapsed_minutes': 0.18627052678333483}
{'step': 120, 'epoch': 0, 'loss': 1.811916712744336, 'backbone_lr': 1.8808777429467086e-05, 'elapsed_minutes': 0.2572471788833354}
{'step': 133, 'epoch': 1, 'validation_macro_f1': 0.0}
{'step': 140, 'epoch': 1, 'loss': 1.461454385910418, 'backbone_lr': 2.19435736677116e-05, 'elapsed_minutes': 0.4171122945000017}
{'step': 160, 'epoch': 1, '

## 8. Official validation evaluation


In [ ]:
def load_ner_checkpoint(path,device):
    state=torch.load(path,map_location='cpu',weights_only=False)
    signature=state['signature']; cfg=signature['config']
    if signature['labels']!=LABELS: raise ValueError('Checkpoint label order mismatch')
    if signature['tokenizer_sha256']!=digest_file(RUN_DIR/'tokenizer.json'): raise ValueError('Checkpoint tokenizer mismatch')
    model=LegalNER(signature['model'],1+2*len(LABELS),cfg,state['model']['class_weights'])
    model.load_state_dict(state['model']); model.to(device).eval()
    return model,signature

def predict_set(checkpoint_path,texts,out_root,cpu=False):
    device=torch.device('cpu' if cpu else 'cuda')
    signature=dict(checkpoint=digest_file(checkpoint_path),
        texts=hashlib.sha256(json.dumps(texts,sort_keys=True,ensure_ascii=False).encode('utf-8')).hexdigest(),
        policy='model4-fixed-teacher-crf-v1')
    tag=hashlib.sha256(json.dumps(signature,sort_keys=True).encode()).hexdigest()[:16]
    out=Path(out_root)/tag; out.mkdir(parents=True,exist_ok=True)
    if (out/'generation.json').exists() and read_json(out/'generation.json')!=signature: raise ValueError('Prediction signature mismatch')
    write_json(out/'generation.json',signature)
    predictions=read_json(out/'prediction.json') if (out/'prediction.json').exists() else {}
    if not set(predictions)<=set(texts): raise ValueError('Unknown saved document IDs')
    validate_predictions(predictions,{k:texts[k] for k in predictions},LABELS)
    model,trained=load_ner_checkpoint(checkpoint_path,device)
    try:
        for key in texts:
            if key in predictions: continue
            data=WindowData(texts,[key],TOKENIZER,trained['config'])
            verify_windows(data,texts)
            result=predict_windows(model,data,LABELS,device)
            validate_predictions(result,{key:texts[key]},LABELS)
            predictions.update(result); write_json(out/'prediction.json',predictions)
            if len(predictions)%10==0: print(f'{len(predictions)}/{len(texts)} documents',flush=True)
        validate_predictions(predictions,texts,LABELS)
        temp=out/'submission.tmp.zip'; final=out/'submission.zip'
        with zipfile.ZipFile(temp,'w',zipfile.ZIP_DEFLATED) as archive: archive.write(out/'prediction.json','prediction.json')
        with zipfile.ZipFile(temp) as archive:
            assert archive.namelist()==['prediction.json'] and archive.testzip() is None
            validate_predictions(json.loads(archive.read('prediction.json')),texts,LABELS)
        os.replace(temp,final)
        return out
    finally:
        del model
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
DEV_OUT=predict_set(RUN_DIR/'best.pt',{k:TEXTS[k] for k in split['val']},RUN_DIR/'validation_predictions')
command=[sys.executable,str(EVALUATOR_PATH),'--truth',str(RUN_DIR/'validation_truth.json'),
    '--prediction',str(DEV_OUT/'prediction.json'),'--labels',str(RUN_DIR/'labels.txt')]
result=subprocess.run(command,capture_output=True,text=True)
print(result.stdout)
if result.stderr: print(result.stderr)
(DEV_OUT/'prof_evaluation.txt').write_text(result.stdout+'\n'+result.stderr,encoding='utf-8')
write_json(DEV_OUT/'evaluator_provenance.json',dict(script_sha256=digest_file(EVALUATOR_PATH),
    truth_sha256=digest_file(RUN_DIR/'validation_truth.json'),prediction_sha256=digest_file(DEV_OUT/'prediction.json'),returncode=result.returncode))
result.check_returncode()
scores=span_metrics({k:GOLD[k] for k in split['val']},read_json(DEV_OUT/'prediction.json'),LABELS)
match=re.search(r'Strict Macro-F1:\s*([0-9.]+)',result.stdout)
assert match and abs(float(match.group(1))-scores['macro_f1'])<1e-6
print('Teacher validation:',TEACHER_STATE['metrics']['macro_f1'])
print('Student validation:',scores['macro_f1'])


Examples: 10
APP: P=0.125000 R=0.054545 F1=0.075949
RESP: P=0.214286 R=0.107143 F1=0.142857
A.COUNSEL: P=0.464286 R=0.245283 F1=0.320988
R.COUNSEL: P=0.250000 R=0.387097 F1=0.303797
JUDGE: P=0.466667 R=0.482759 F1=0.474576
WIT: P=0.437500 R=0.466667 F1=0.451613
AUTH: P=0.323741 R=0.450000 F1=0.376569
COURT: P=0.786885 R=0.813559 F1=0.800000
STAT: P=0.531532 R=0.572816 F1=0.551402
PREC: P=0.096774 R=0.095541 F1=0.096154
DATE: P=0.902954 R=0.918455 F1=0.910638
CASENO: P=0.652174 R=0.642857 F1=0.647482
Strict Macro-F1:        0.429335
Strict Macro-Precision: 0.437650
Strict Macro-Recall:    0.436393

Teacher validation: 0.5492515376309127
Student validation: 0.4293354893553228


## 9. Test submission


In [ ]:
import base64
TEST_OUT=predict_set(RUN_DIR/'best.pt',TEST_TEXTS,RUN_DIR/'test_predictions')
submission=TEST_OUT/'submission.zip'
payload=base64.b64encode(submission.read_bytes()).decode('ascii')
print(submission)
display(HTML(f'<a download="model4_distillation_submission.zip" href="data:application/zip;base64,{payload}">Download model 4 submission</a>'))


/kaggle/working/phase2_model4_distillation/student/test_predictions/c4b8dc51a623373c/submission.zip


Download model 4 submission